# Inception v3 trên COVID-QU-Ex: COVID-19 vs Normal (mạng trích đặc trưng cho FID)

- Dữ liệu: `dacntt-kltn/covid-normal-cxr` (train / validation / test), nhãn `normal=0`, `covid19=1`
- Code: clone từ GitHub `Vinh306/medical-image-classification-counterfactual-explanation` (thư mục `cGAN`), dùng **đúng** `build_inception_classifier` và `prepare_for_inception` của repo để tiền xử lý lúc huấn luyện trùng với lúc tính FID
- Mô hình: Inception v3 (khởi tạo ImageNet, tùy chọn), 1 logit
- **Early stopping** theo tập val, **checkpoint đẩy lên Hugging Face** (đồng bộ nền + lần cuối)
- Đánh giá trên tập **test** bằng checkpoint tốt nhất theo val
- Xuất `inception_cxr.pt` (state_dict thuần) để dùng với `test.py --fid_net` của repo

## 1. Import

In [ ]:
import os, re, sys, json, time, shutil, threading, subprocess
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # tắt thanh tiến trình khi tải dữ liệu
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)

from datasets import load_dataset
from huggingface_hub import login, HfApi, create_repo, list_repo_files, hf_hub_download
from kaggle_secrets import UserSecretsClient

## 2. Cấu hình

In [ ]:
# Hugging Face
HF_USERNAME = "dacntt-kltn"
DATASET_REPO_ID = f"{HF_USERNAME}/covid-normal-cxr"
MODEL_REPO_ID = f"{HF_USERNAME}/inception-v3-covid-normal-cxr"
MODEL_PRIVATE = False
RUNS_PREFIX = "training-runs"
RUN_SUFFIX = "inception-v3-covid-normal"

# Code trên GitHub
GITHUB_REPO_URL = "https://github.com/Vinh306/medical-image-classification-counterfactual-explanation"
CODE_DIR = Path("/kaggle/working/cf-explanation-code")
CODE_SUBDIR = "cGAN"

# Tham số huấn luyện
SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 100
LR_PATIENCE = 2
USE_PRETRAINED = True # True: khởi tạo từ ImageNet, False: huấn luyện từ đầu

# Early stopping
MONITOR = "val_loss" # chỉ số theo dõi để dừng
PATIENCE = 10
MIN_DELTA = 1e-4 # mức cải thiện tối thiểu
MODE = "min" if MONITOR == "val_loss" else "max"

# Đồng bộ lên Hub
RESUME_FROM_HUB = False 
SYNC_INTERVAL_SEC = 300 # đẩy checkpoint lên Hub mỗi 5 phút

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("device:", device, "| AMP:", use_amp)

## 3. Lấy code từ GitHub

In [ ]:
if CODE_DIR.exists():
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(CODE_DIR)], check=True)

GIT_COMMIT = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"],
                            capture_output=True, text=True, check=True).stdout.strip()
print("Commit đang dùng:", GIT_COMMIT)

sys.path.insert(0, str(CODE_DIR / CODE_SUBDIR))
import common, metrics
assert str(CODE_DIR) in common.__file__ and str(CODE_DIR) in metrics.__file__, \
    "Import nhầm module trùng tên, không phải code trong repo"

from common import seed_everything, IMG_SIZE, LABELS
from metrics import build_inception_classifier, prepare_for_inception
assert len(LABELS) == 1, f"Repo cần có LABELS = ['covid19'] (1 logit), hiện tại: {LABELS}"

seed_everything(SEED)
print("Đã import từ", CODE_DIR / CODE_SUBDIR, "| IMG_SIZE =", IMG_SIZE, "| LABELS =", LABELS)

## 4. Đăng nhập Hugging Face và tạo repo model

In [ ]:
token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)

api = HfApi()
print("Đăng nhập là:", api.whoami()["name"])
create_repo(MODEL_REPO_ID, repo_type="model", private=MODEL_PRIVATE, exist_ok=True)
print(f"Model repo: https://huggingface.co/{MODEL_REPO_ID}")
print(f"Dataset repo: https://huggingface.co/datasets/{DATASET_REPO_ID}")

## 5. Tải dữ liệu từ Hugging Face

In [ ]:
ds = load_dataset(DATASET_REPO_ID, token=token)
print(ds)

label_names = ds["train"].features["label"].names
print("Nhãn:", label_names)
assert label_names == ["normal", "covid19"], "Thứ tự nhãn không đúng (cần normal=0, covid19=1)"

for split in ds:
    cnt = pd.Series(ds[split]["label"]).value_counts().sort_index()
    print(f"{split:>10}: " + ", ".join(f"{label_names[i]}={n}" for i, n in cnt.items()))

im = ds["train"][0]["image"]
print("Ảnh mẫu:", im.size, im.mode)
assert im.size == (IMG_SIZE, IMG_SIZE), \
    f"Ảnh {im.size} khác {IMG_SIZE}x{IMG_SIZE}: cần thêm bước crop/resize giống load_image của repo"

## 6. Tiền xử lý và DataLoader

In [ ]:
train_tf = T.Compose([
    T.Grayscale(1),
    T.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    T.ToTensor(),
])
eval_tf = T.Compose([T.Grayscale(1), T.ToTensor()])

def make_fn(tf):
    def fn(batch):
        return {"pixel_values": [tf(im) for im in batch["image"]], "label": batch["label"]}
    return fn

def collate(batch):
    x = torch.stack([b["pixel_values"] for b in batch])
    y = torch.tensor([b["label"] for b in batch], dtype=torch.float32)
    return x, y

g = torch.Generator(); g.manual_seed(SEED)
train_dl = DataLoader(ds["train"].with_transform(make_fn(train_tf)), batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True, generator=g)
val_dl = DataLoader(ds["validation"].with_transform(make_fn(eval_tf)), batch_size=64, shuffle=False,
                    num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)
test_dl = DataLoader(ds["test"].with_transform(make_fn(eval_tf)), batch_size=64, shuffle=False,
                     num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)

x, y = next(iter(train_dl))
print("batch:", tuple(x.shape), tuple(y.shape), "| nhãn trong batch:", y.int().tolist()[:16])
print(f"khoảng giá trị ảnh: [{x.min().item():.3f}, {x.max().item():.3f}]")
assert x.shape[1:] == (1, IMG_SIZE, IMG_SIZE) and x.min() >= 0 and x.max() <= 1

## 7. Mô hình và hàm đánh giá

In [ ]:
def build_model():
    return build_inception_classifier(pretrained=USE_PRETRAINED)

def forward_logits(model, x01):
    return model(prepare_for_inception(x01)).squeeze(1)

criterion = nn.BCEWithLogitsLoss()

@torch.no_grad()
def evaluate(model, dl, threshold=0.5):
    model.eval()
    loss_sum, probs, labels = 0.0, [], []
    for x, y in dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = forward_logits(model, x)
        logits = logits.float()
        loss_sum += criterion(logits, y).item() * len(y)
        probs.append(torch.sigmoid(logits).cpu()); labels.append(y.cpu())
    probs = torch.cat(probs).numpy(); labels = torch.cat(labels).numpy().astype(int)
    pred = (probs >= threshold).astype(int)
    return dict(loss=loss_sum / len(labels), acc=accuracy_score(labels, pred),
                auc=roc_auc_score(labels, probs), f1=f1_score(labels, pred),
                probs=probs, labels=labels, pred=pred)

def atomic_save(obj, path):
    tmp = str(path) + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

## 8. Thư mục run và resume từ Hub

In [ ]:
LOCAL_OUTDIR = Path("/kaggle/working/inception-v3-training-runs")
LOCAL_OUTDIR.mkdir(parents=True, exist_ok=True)

existing_files = list_repo_files(MODEL_REPO_ID, repo_type="model")
run_re = re.compile(rf"^{RUNS_PREFIX}/([0-9]{{5}}-[^/]+)/(.+)$")
hub_runs = {}
for f in existing_files:
    m = run_re.match(f)
    if m:
        hub_runs.setdefault(m.group(1), set()).add(m.group(2))
print("Các run đã có trên Hub:", sorted(hub_runs))

resume_state, RUN_NAME = None, None
if RESUME_FROM_HUB and hub_runs:
    latest = sorted(hub_runs)[-1]
    if "last.pt" in hub_runs[latest] and "finished.json" not in hub_runs[latest]:
        RUN_NAME = latest
        RUN_DIR = LOCAL_OUTDIR / RUN_NAME
        RUN_DIR.mkdir(parents=True, exist_ok=True)
        for fn in ["last.pt", "best.pt", "history.csv", "config.json"]:
            if fn in hub_runs[latest]:
                p = hf_hub_download(MODEL_REPO_ID, f"{RUNS_PREFIX}/{latest}/{fn}", repo_type="model")
                shutil.copy(p, RUN_DIR / fn)
        resume_state = torch.load(RUN_DIR / "last.pt", map_location="cpu", weights_only=False)
        print(f"RESUME run {RUN_NAME} từ epoch {resume_state['state']['epoch']}")

if resume_state is None:
    next_id = int(sorted(hub_runs)[-1][:5]) + 1 if hub_runs else 0
    RUN_NAME = f"{next_id:05d}-{RUN_SUFFIX}"
    RUN_DIR = LOCAL_OUTDIR / RUN_NAME
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    (RUN_DIR / "config.json").write_text(json.dumps(dict(
        model="inception_v3", pretrained_imagenet=USE_PRETRAINED, dataset=DATASET_REPO_ID,
        seed=SEED, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY, max_epochs=MAX_EPOCHS,
        lr_patience=LR_PATIENCE, monitor=MONITOR, patience=PATIENCE, min_delta=MIN_DELTA,
        code_repo=GITHUB_REPO_URL, code_commit=GIT_COMMIT), indent=2))
    print("Train từ đầu, run mới:", RUN_NAME)

## 9. Huấn luyện + early stopping + đồng bộ Hub

In [ ]:
model = build_model().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode=MODE, factor=0.5, patience=LR_PATIENCE)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

state = dict(epoch=0, best=None, best_epoch=-1, bad_epochs=0, history=[])
if resume_state is not None:
    model.load_state_dict(resume_state["model"])
    opt.load_state_dict(resume_state["opt"])
    sched.load_state_dict(resume_state["sched"])
    scaler.load_state_dict(resume_state["scaler"])
    state = resume_state["state"]

def is_better(cur, best):
    if best is None: return True
    return cur < best - MIN_DELTA if MODE == "min" else cur > best + MIN_DELTA

# Luồng đồng bộ nền
api_sync = HfApi()
sync_stop = threading.Event()

def upload_run_dir(msg):
    api_sync.upload_folder(folder_path=str(RUN_DIR), repo_id=MODEL_REPO_ID, repo_type="model",
                           path_in_repo=f"{RUNS_PREFIX}/{RUN_NAME}", ignore_patterns=["*.tmp"],
                           commit_message=msg)

def sync_to_hub():
    last_sig = None
    while not sync_stop.wait(SYNC_INTERVAL_SEC):
        try:
            sig = tuple(sorted((p.name, p.stat().st_mtime) for p in RUN_DIR.iterdir()
                               if p.is_file() and p.suffix != ".tmp"))
            if sig and sig != last_sig:
                upload_run_dir(f"sync {RUN_NAME}")
                last_sig = sig
                print("[sync] đã đồng bộ lên Hub.")
        except Exception as e:
            print("[sync] Lỗi khi đồng bộ:", e)

sync_thread = threading.Thread(target=sync_to_hub, daemon=True)
sync_thread.start()

# Vòng lặp huấn luyện
stop_reason = None
try:
    for epoch in range(state["epoch"], MAX_EPOCHS):
        t0 = time.time()
        model.train()
        loss_sum, n = 0.0, 0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = forward_logits(model, x)
            loss = criterion(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            loss_sum += loss.item() * len(y); n += len(y)

        val = evaluate(model, val_dl)
        row = dict(epoch=epoch + 1, train_loss=loss_sum / n, val_loss=val["loss"], val_acc=val["acc"],
                   val_auc=val["auc"], val_f1=val["f1"], lr=opt.param_groups[0]["lr"])
        state["history"].append(row)
        cur = row[MONITOR]
        sched.step(cur)

        if is_better(cur, state["best"]):
            state.update(best=cur, best_epoch=epoch + 1, bad_epochs=0)
            atomic_save(dict(model=model.state_dict(), epoch=epoch + 1, monitor=MONITOR, value=cur,
                             label_names=label_names), RUN_DIR / "best.pt")
            tag = "  <- best"
        else:
            state["bad_epochs"] += 1
            tag = f"  (không cải thiện {state['bad_epochs']}/{PATIENCE})"

        state["epoch"] = epoch + 1
        atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                         scaler=scaler.state_dict(), state=state), RUN_DIR / "last.pt")
        pd.DataFrame(state["history"]).to_csv(RUN_DIR / "history.csv", index=False)

        print(f"epoch {epoch+1:02d} | train_loss {row['train_loss']:.4f} | val_loss {row['val_loss']:.4f} | "
              f"val_acc {row['val_acc']:.4f} | val_auc {row['val_auc']:.4f} | lr {row['lr']:.1e} | "
              f"{time.time()-t0:.0f}s{tag}")

        if state["bad_epochs"] >= PATIENCE:
            stop_reason = "early_stopping"
            print(f"Early stopping: {MONITOR} không cải thiện sau {PATIENCE} epoch. Tốt nhất ở epoch {state['best_epoch']}.")
            break
    else:
        stop_reason = "max_epochs"

    if stop_reason:   # chỉ đánh dấu đã xong khi vòng lặp kết thúc bình thường
        (RUN_DIR / "finished.json").write_text(json.dumps(dict(
            reason=stop_reason, best_epoch=state["best_epoch"], monitor=MONITOR, best_value=state["best"],
            epochs_trained=state["epoch"]), indent=2))
finally:
    sync_stop.set()
    sync_thread.join(timeout=5)
    upload_run_dir(f"final {RUN_NAME}")
    print(f"Đã đẩy kết quả lên Hugging Face: https://huggingface.co/{MODEL_REPO_ID}/tree/main/{RUNS_PREFIX}/{RUN_NAME}")

## 10. Đường cong huấn luyện

In [ ]:
h = pd.DataFrame(state["history"])
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val"); ax[0].set_title("Loss")
ax[1].plot(h.epoch, h.val_acc); ax[1].set_title("Val accuracy")
ax[2].plot(h.epoch, h.val_auc); ax[2].set_title("Val AUC")
for a in ax:
    a.axvline(state["best_epoch"], color="gray", ls="--", lw=1); a.set_xlabel("epoch")
ax[0].legend(); plt.tight_layout()
plt.savefig(RUN_DIR / "curves.png", dpi=300); plt.show()

## 11. Kiểm tra trên tập test

Dùng checkpoint **tốt nhất theo val** (`best.pt`).
Quy ước: lớp dương = `covid19`; sensitivity = recall của covid19, specificity = recall của normal.

In [ ]:
def run_test(model, out_dir, tag="test"):
    r = evaluate(model, test_dl)
    cm = confusion_matrix(r["labels"], r["pred"])
    tn, fp, fn, tp = cm.ravel()
    metrics_out = dict(accuracy=r["acc"], auc=r["auc"], f1=r["f1"], loss=r["loss"],
                       sensitivity=tp / (tp + fn), specificity=tn / (tn + fp), n=int(len(r["labels"])),
                       confusion_matrix=cm.tolist(), label_names=label_names)
    print(classification_report(r["labels"], r["pred"], target_names=label_names, digits=4))
    print({k: round(v, 4) for k, v in metrics_out.items() if isinstance(v, float)})

    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(label_names); ax.set_yticks([0, 1]); ax.set_yticklabels(label_names)
    ax.set_xlabel("dự đoán"); ax.set_ylabel("thực tế")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.tight_layout(); plt.savefig(Path(out_dir) / f"{tag}_confusion_matrix.png", dpi=120); plt.show()
    (Path(out_dir) / f"{tag}_metrics.json").write_text(json.dumps(metrics_out, indent=2, ensure_ascii=False))
    return metrics_out

best_ckpt = torch.load(RUN_DIR / "best.pt", map_location="cpu", weights_only=False)
print(f"Nạp best.pt: epoch {best_ckpt['epoch']}, {best_ckpt['monitor']} = {best_ckpt['value']:.4f}")
model = build_inception_classifier(pretrained=False).to(device)   # nạp trọng số của best.pt nên không cần tải ImageNet
model.load_state_dict(best_ckpt["model"])
test_metrics = run_test(model, RUN_DIR)

## 12. Đẩy kết quả test lên Hub

In [ ]:
upload_run_dir(f"test results {RUN_NAME}")
print(f"Xong: https://huggingface.co/{MODEL_REPO_ID}/tree/main/{RUNS_PREFIX}/{RUN_NAME}")

## 13. Xuất checkpoint cho FID (`inception_cxr.pt`)

In [ ]:
from metrics import build_fid_net, inception_features

FID_CKPT = RUN_DIR / "inception_cxr.pt"
atomic_save({k: v.detach().cpu() for k, v in best_ckpt["model"].items()}, FID_CKPT)
print("Đã lưu:", FID_CKPT, f"({FID_CKPT.stat().st_size / 1e6:.0f} MB)")

# Kiểm tra bằng code của repo (fp32, không autocast)
fid_net = build_fid_net(str(FID_CKPT), device=device)
x_chk, _ = next(iter(test_dl))
x_chk = x_chk[:16].to(device)
with torch.no_grad():
    feats = inception_features(fid_net, x_chk)
    logits_ref = model.eval()(prepare_for_inception(x_chk)).squeeze(1)
    logits_from_feats = model.fc(feats).squeeze(1)
diff = (logits_ref - logits_from_feats).abs().max().item()
print("Kích thước đặc trưng:", tuple(feats.shape), "| sai khác logit tối đa:", f"{diff:.2e}")
assert feats.shape[1] == 2048 and diff < 1e-3, "Checkpoint FID không khớp với model đã huấn luyện"

upload_run_dir(f"export fid net {RUN_NAME}")
print(f"Đã đẩy lên: https://huggingface.co/{MODEL_REPO_ID}/tree/main/{RUNS_PREFIX}/{RUN_NAME}")

## 14. Test lại một checkpoint bất kỳ trên Hub

In [ ]:
HF_CHECKPOINT_FILENAME = ""

if not HF_CHECKPOINT_FILENAME:
    files = list_repo_files(MODEL_REPO_ID, repo_type="model")
    bests = sorted(f for f in files if f.startswith(f"{RUNS_PREFIX}/") and f.endswith("/best.pt"))
    HF_CHECKPOINT_FILENAME = bests[-1]
print("Dùng checkpoint:", HF_CHECKPOINT_FILENAME)

p = hf_hub_download(MODEL_REPO_ID, HF_CHECKPOINT_FILENAME, repo_type="model")
ck = torch.load(p, map_location="cpu", weights_only=False)
m = build_inception_classifier(pretrained=False)
m.load_state_dict(ck["model"]); m = m.to(device)
Path("/kaggle/working/hub_test").mkdir(exist_ok=True)
_ = run_test(m, "/kaggle/working/hub_test", tag="hub_ckpt")